In [8]:
# Import packages.
import pandas as pd
import numpy as np
import kagglehub
import os
from google.colab import drive

# Download data.
path = kagglehub.dataset_download("netflix-inc/netflix-prize-data")
print("Path to dataset files:", path)


# Mount drive if you plan to use your google drive.
drive.mount('/content/drive')

Using Colab cache for faster access to the 'netflix-prize-data' dataset.
Path to dataset files: /kaggle/input/netflix-prize-data
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import io # Added for StringIO

def load_netflix_data_file(filepath):
  # Read all lines first, which can be faster than line-by-line I/O in a loop
  with open(filepath, 'r') as f:
    lines = f.readlines()

  all_data_frames = []
  current_movie_id = None
  movie_ratings_block = []

  for line in lines:
    line = line.strip()

    if line.endswith(':'):
      # If a previous block of ratings exists, process it
      if movie_ratings_block:
        # Use StringIO to treat the block as a file for pd.read_csv
        data_io = io.StringIO("\n".join(movie_ratings_block))
        # Read the block using pandas, much faster than manual parsing
        temp_df = pd.read_csv(data_io, header=None, names=['UserID', 'Rating', 'Date'], dtype={'UserID': int, 'Rating': int})
        temp_df['MovieID'] = current_movie_id
        all_data_frames.append(temp_df)

      # Update current movie ID and reset block for next movie
      current_movie_id = int(line[:-1])
      movie_ratings_block = []
    else:
      # Accumulate rating lines for the current movie
      movie_ratings_block.append(line)

  # Process the very last block of ratings after the loop finishes
  if movie_ratings_block:
    data_io = io.StringIO("\n".join(movie_ratings_block))
    temp_df = pd.read_csv(data_io, header=None, names=['UserID', 'Rating', 'Date'], dtype={'UserID': int, 'Rating': int})
    temp_df['MovieID'] = current_movie_id
    all_data_frames.append(temp_df)

  # Concatenate all DataFrames and perform vectorized date conversion
  if all_data_frames:
    final_df = pd.concat(all_data_frames, ignore_index=True)
    final_df['Date'] = pd.to_datetime(final_df['Date'])
    # Reorder columns to match original intent
    return final_df[['MovieID', 'UserID', 'Rating', 'Date']]
  else:
    return pd.DataFrame(columns=['MovieID', 'UserID', 'Rating', 'Date'])

In [19]:
combined_data_filenames = [
    'combined_data_1.txt',
    'combined_data_2.txt',
    'combined_data_3.txt',
    'combined_data_4.txt'
]

lists_of_dfs = []

for filename in combined_data_filenames:
  full_filepath = os.path.join(path, filename)
  print(f"Loading data from: {full_filepath}")

  df_single_file = load_netflix_data_file(full_filepath)

  lists_of_dfs.append(df_single_file)

final_ratings_df = pd.concat(lists_of_dfs, ignore_index = True)

print("\nFinal combined DataFrame info:")
final_ratings_df.info()

print("\nFirst 5 rows of the final combined DataFrame:")
print(final_ratings_df.head())

Loading data from: /kaggle/input/netflix-prize-data/combined_data_1.txt
Loading data from: /kaggle/input/netflix-prize-data/combined_data_2.txt
Loading data from: /kaggle/input/netflix-prize-data/combined_data_3.txt
Loading data from: /kaggle/input/netflix-prize-data/combined_data_4.txt

Final combined DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100480507 entries, 0 to 100480506
Data columns (total 4 columns):
 #   Column   Dtype         
---  ------   -----         
 0   MovieID  int64         
 1   UserID   int64         
 2   Rating   int64         
 3   Date     datetime64[ns]
dtypes: datetime64[ns](1), int64(3)
memory usage: 3.0 GB

First 5 rows of the final combined DataFrame:
   MovieID   UserID  Rating       Date
0        1  1488844       3 2005-09-06
1        1   822109       5 2005-05-13
2        1   885013       4 2005-10-19
3        1    30878       4 2005-12-26
4        1   823519       3 2004-05-03


In [27]:
csv_path = ("/kaggle/input/netflix-prize-data/movie_titles.csv")
movie_title_df = pd.read_csv(csv_path, encoding='latin-1', engine='python', header=None, names=['MovieID', 'YearOfRelease', 'Title'], on_bad_lines='skip')

movie_title_df.head()

,MovieID,YearOfRelease,Title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
